# k-Nearest Neighbours, Support Vector Machines and Naive Bayes – Titanic

**Dataset:** Titanic – 891 passengers (Kaggle), a classic classification example. File: `../00_Data/titanic.csv`.

**Question:** Can the survival of a Titanic passenger be predicted from a few characteristics with distance-based
methods (k-NN, SVM) and with Naive Bayes?

**After this exercise you can ...**
- apply k-NN, explain why features must be scaled and choose $k$ by cross-validation,
- explain the role of the margin, the support vectors, $C$ and the kernel of an SVM,
- compute a Naive Bayes classification by hand and apply Naive Bayes to categorical data.

## Concept

The concepts behind this exercise (formulas and worked examples) are
explained in the notebook [knn_naive_bayes_svm_concept.ipynb](knn_naive_bayes_svm_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import CategoricalNB
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
)
from sklearn.metrics import confusion_matrix
from sklearn.inspection import DecisionBoundaryDisplay

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-NN and Naive Bayes by hand

*Time: about 15 minutes. Use the notebook cell below as your calculator: type the formulas with the numbers in
Python (e.g. `(3 + 5) / 2` or `0.4 ** 2`) – no paper needed.*

**a) k-NN with and without scaling.** A new passenger is 30 years old and paid a fare of 40 pounds. Two known
passengers:

| passenger | survived? | age [years] | fare [£] |
|---|---|---|---|
| P1 | no | 58 | 41 |
| P2 | yes | 29 | 70 |

1. Compute the Euclidean distances $\sqrt{\Delta\text{age}^2 + \Delta\text{fare}^2}$ of the new passenger to P1 and P2.
   Who is the nearest neighbour ($k = 1$)?
2. Standardise the differences: divide $\Delta$age by 13 and $\Delta$fare by 50 (the standard deviations) and compute
   the distances again. Who is the nearest neighbour now? Why does the result change?

**b) Naive Bayes.** Of the 891 passengers, 342 survived and 549 died. Among the survivors there were 233 women and 136
first-class passengers; among those who died there were 81 women and 80 first-class passengers. Compute the Naive Bayes
scores for a **woman in 1st class**:

score(survived) = P(survived) · P(woman | survived) · P(1st class | survived), score(died) analogously.

Her probability of survival is score(survived) / (score(survived) + score(died)). Compare it with the observed survival
rate of women in 1st class (91 of 94).

In [ ]:
# a) Distances of the new passenger (age 30, fare 40) to
#    P1 (age 58, fare 41) and P2 (age 29, fare 70) on the original scale
d1 = np.sqrt((58 - 30) ** 2 + (41 - 40) ** 2)
d2 = np.sqrt((29 - 30) ** 2 + (70 - 40) ** 2)
print('Original scale: P1 =', round(d1, 1), ' P2 =', round(d2, 1))

# Distances with standardised differences (age / 13, fare / 50)
d1_std = np.sqrt(((58 - 30) / 13) ** 2 + ((41 - 40) / 50) ** 2)
d2_std = np.sqrt(((29 - 30) / 13) ** 2 + ((70 - 40) / 50) ** 2)
print('Standardised:   P1 =', round(d1_std, 3), ' P2 =', round(d2_std, 3))

# b) Naive Bayes scores and probability of survival of a woman in 1st class
score_surv = 342 / 891 * 233 / 342 * 136 / 342
score_died = 549 / 891 * 81 / 549 * 80 / 549
print('score(survived) =', round(score_surv, 4))
print('score(died) =', round(score_died, 4))
print('P(survived) =', round(score_surv / (score_surv + score_died), 3))
print('Observed survival rate =', round(91 / 94, 3))

**Solution:**

a) 1. Original scale: P1 ≈ 28.0, P2 ≈ 30.0 → nearest neighbour P1 → **died**.

2. Standardised: P1 = $\sqrt{(28/13)^2 + (1/50)^2} \approx 2.154$; P2 = $\sqrt{(1/13)^2+(30/50)^2} \approx 0.605$ →
nearest neighbour P2 → **survived**. On the original scale one year of age counts as much as one pound of fare.
Because ages differ less than fares (standard deviation 13 vs. 50), a 28-year age difference (P1) looks "close". After
standardising, both features count in units of their typical spread, and P1 (58 years old) is far away.

b) score(survived) = $\frac{342}{891}\cdot\frac{233}{342}\cdot\frac{136}{342} \approx 0.104$;
score(died) = $\frac{549}{891}\cdot\frac{81}{549}\cdot\frac{80}{549} \approx 0.0132$ → predict **survived**;
P(survived | woman, 1st class) $= 0.104/(0.104+0.0132) \approx 0.89$. The observed survival rate is even higher (91/94
≈ 0.97): sex and class are not independent given survival – the "naive" assumption leads to a biased probability, but
the predicted class is correct.

## Exercise 2: Prepare the Titanic data

a) Import the Titanic data and prepare the features `male`, `Age` (missing values replaced by the median), `Pclass`,
`SibSp`, `Parch` and `Fare`. Split the data into a stratified training (80 %) and test set (20 %) with `random_state=42`.

b) Create a scatter plot of `Age` against `Fare` (logarithmic y-axis), coloured by `Survived`, and compare the scales of
the features.

In [ ]:
# a) Import the data as titanic and prepare the features
titanic = pd.read_csv('../00_Data/titanic.csv', sep=',')
titanic['male'] = (titanic['Sex'] == 'male').astype(int)
titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())
features = ['male', 'Age', 'Pclass', 'SibSp', 'Parch', 'Fare']
X = titanic[features]
y = titanic['Survived']

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('Training set:', X_train.shape, ' Test set:', X_test.shape)
print(X_train.describe().round(1))

# b) Scatter plot Age vs. Fare (log scale; +1 because some fares are 0)
died = titanic[titanic['Survived'] == 0]
survivors = titanic[titanic['Survived'] == 1]
plt.scatter(died['Age'], died['Fare'] + 1, label='died')
plt.scatter(survivors['Age'], survivors['Fare'] + 1, label='survived')
plt.yscale('log')
plt.xlabel('Age [years]')
plt.ylabel('Fare + 1 [£] (log scale)')
plt.legend()
plt.show()

**Solution:**

The features are on very different scales: `Fare` ranges from 0 to 512 £ (standard deviation ≈ 50), `Age` from 0 to 80
years (≈ 13), `male` is 0/1 and `Pclass` 1–3. Survivors are more frequent among passengers with high fares and among
children; in the age–fare plane the two classes overlap strongly – sex, the most important feature, is not visible in
this plot.

## Exercise 3: k-NN – the effect of scaling and the choice of k

a) Compute the 5-fold CV accuracy (on the training set) of `KNeighborsClassifier(n_neighbors=5)` on the **unscaled**
features and of `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`. Explain the difference.

b) Compute the 5-fold CV accuracy of the scaled k-NN for $k = 1, 2, \dots, 40$ and plot it against $k$. Choose $k$ and
compute accuracy and confusion matrix on the test set.

c) Predict the survival probability of a 30-year-old man in 3rd class without relatives (fare 8 £) and of a
30-year-old woman in 1st class without relatives (fare 80 £).

In [ ]:
# Exercise 3: 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# a) k-NN without and with scaling (the scaler is part of the pipeline)
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_scaled = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=5)
)
acc_raw = cross_val_score(knn_raw, X_train, y_train, cv=cv)
acc_scaled = cross_val_score(knn_scaled, X_train, y_train, cv=cv)
print('Unscaled: CV accuracy =', round(acc_raw.mean(), 3))
print('Scaled:   CV accuracy =', round(acc_scaled.mean(), 3))

# b) CV accuracy of the scaled k-NN for k = 1, ..., 40; best k on the test set
ks = range(1, 41)
cv_acc = []
for k in ks:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    cv_acc.append(cross_val_score(knn, X_train, y_train, cv=cv).mean())

# Figure showing the CV accuracy by k
plt.plot(ks, cv_acc, marker='o')
plt.xlabel('k (number of neighbours)')
plt.ylabel('5-fold CV accuracy')
plt.show()

# k with the best CV accuracy: fit and evaluate on the test set
best_k = ks[np.argmax(cv_acc)]
print('Chosen k:', best_k, ' CV accuracy:', round(max(cv_acc), 3))
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k))
knn.fit(X_train, y_train)
print('Test accuracy:', round(knn.score(X_test, y_test), 3))
print(confusion_matrix(y_test, knn.predict(X_test)))

# c) Two new passengers: man in 3rd class, woman in 1st class
new_passengers = pd.DataFrame(
    {
        'male': [1, 0],
        'Age': [30, 30],
        'Pclass': [3, 1],
        'SibSp': [0, 0],
        'Parch': [0, 0],
        'Fare': [8, 80],
    }
)

# Probabilities (columns: died, survived)
print(knn.predict_proba(new_passengers).round(2))

**Solution:**

a) Without scaling the CV accuracy is only about **0.71**, with scaling about **0.82**. Without scaling, the distance is
dominated by `Fare` (differences of tens or hundreds of pounds) and `Age`; the most important feature `male` (0/1) and the
class (1–3) hardly contribute to the distance – exactly the effect of Exercise 1a. The scaler must be part of the
pipeline so that it is fitted on the training folds only (no data leakage).

b) The CV accuracy varies only between about 0.80 and 0.83 for most $k$; CV selects $k = 3$ (CV accuracy ≈ 0.83, test
accuracy ≈ 0.80). Small $k$ give flexible, larger $k$ smoother decision boundaries.

c) With $k = 3$ all three neighbours of the 30-year-old man in 3rd class died, and all three neighbours of the woman in
1st class survived → probabilities of 0 and 1. With small $k$, k-NN probabilities are very coarse (only 0, 1/3, 2/3, 1);
a larger $k$ gives finer probabilities.

## Exercise 4: Support vector machines – C and kernel

a) Compute the 5-fold CV accuracy and the number of support vectors of
`make_pipeline(StandardScaler(), SVC(kernel=..., C=...))` for `kernel` ∈ {`'linear'`, `'rbf'`} and
$C$ ∈ {0.01, 1, 100} on the training set (all six features).

b) To *see* the boundaries, fit the same six models on only the two features `Age` and `log_fare = log(1 + Fare)` and
plot the decision regions in a 2 × 3 grid (the code for the grid is given), with the support vectors circled.

c) Describe the influence of $C$ and of the kernel. Which model would you choose?

In [ ]:
# a) CV accuracy and number of support vectors for each kernel and C
for kernel in ['linear', 'rbf']:
    for C in [0.01, 1, 100]:
        svm = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=C))
        acc = cross_val_score(svm, X_train, y_train, cv=cv).mean()
        svm.fit(X_train, y_train)
        n_sv = len(svm[-1].support_)  # svm[-1] = the SVC in the pipeline
        print(
            'kernel =',
            kernel,
            ' C =',
            C,
            ' support vectors:',
            n_sv,
            ' CV accuracy:',
            round(acc, 3),
        )

# b) Only two features: Age and log(1 + Fare)
X2_train = pd.DataFrame(
    {'Age': X_train['Age'], 'log_fare': np.log1p(X_train['Fare'])}
)

# Figure showing the decision regions for each kernel and C (2 x 3 plots)
plt.figure(figsize=(16, 9))
i = 1
for kernel in ['linear', 'rbf']:
    for C in [0.01, 1, 100]:
        svm = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=C))
        svm.fit(X2_train, y_train)
        ax = plt.subplot(2, 3, i)
        DecisionBoundaryDisplay.from_estimator(svm, X2_train, alpha=0.3, ax=ax)
        plt.scatter(X2_train['Age'], X2_train['log_fare'], c=y_train, s=8)

        # Circle the support vectors
        sv = X2_train.iloc[svm[-1].support_]
        plt.scatter(
            sv['Age'], sv['log_fare'], s=40, facecolors='none', edgecolors='k'
        )
        plt.title(kernel + ', C = ' + str(C) + ': ' + str(len(sv)) + ' SV')
        i = i + 1
plt.show()

**Solution:**

a) and b):
- **Small $C$ (0.01):** very wide margin, many training errors are tolerated → almost all points are support vectors. With
  the RBF kernel the model predicts "died" for everybody (CV accuracy ≈ 0.62 = share of non-survivors, 547 of 712 points
  are support vectors) → **underfitting**.
- **$C = 1$:** the RBF SVM reaches the best CV accuracy (≈ 0.82, similar to k-NN); the linear SVM ≈ 0.79.
- **Large $C$ (100):** narrow margin, the model tries to classify every training point correctly. With the RBF kernel the
  boundary in the age–fare plot becomes very wiggly ("islands" around single passengers) and the CV accuracy decreases
  slightly (≈ 0.82) → beginning **overfitting**.
- **Kernel:** the linear kernel gives straight boundaries – for the linear SVM, $C$ hardly matters here (≈ 0.79 for all
  values). The RBF kernel can model interactions (e.g. children survive more often regardless of the fare), which gives
  better results.

c) An RBF SVM with $C = 1$. $C$ (and the kernel width `gamma`) should be tuned with cross-validation. The two-feature plot
is only for illustration – without the feature `male` the classes overlap strongly.

## Exercise 5: Naive Bayes with categorical features

a) Fit a `CategoricalNB(alpha=1e-10)` (practically no smoothing) on the two features `male` and `Pclass` of **all**
passengers and predict the survival probability of a woman in 1st class. Compare with your hand calculation of
Exercise 1b.

b) Compute the 5-fold CV accuracy of a `CategoricalNB()` with the features `male`, `Pclass` and `age_group`
(0 = child under 10, 1 = 10–59 years, 2 = 60 years and older) on the training set. Compare with k-NN and SVM.

In [ ]:
# a) CategoricalNB on male and Pclass (all passengers)
X_cat = pd.DataFrame(
    {'male': titanic['male'], 'Pclass': titanic['Pclass'] - 1}
)  # categories must start at 0
nb_cat = CategoricalNB(alpha=1e-10)
nb_cat.fit(X_cat, titanic['Survived'])

# Probability of survival of a woman (male = 0) in 1st class (Pclass = 0)
woman_1st = pd.DataFrame({'male': [0], 'Pclass': [0]})
p_woman = nb_cat.predict_proba(woman_1st)[0, 1]
print('P(survived | woman, 1st class) =', round(p_woman, 3))

# b) CategoricalNB with the features male, Pclass and age_group

# Categorical features of the training set: male, class (0-2) and
# age group (0 = under 10, 1 = 10-59, 2 = 60 and older)
X_cat_train = pd.DataFrame(
    {
        'male': X_train['male'],
        'Pclass': X_train['Pclass'] - 1,
        'age_group': pd.cut(
            X_train['Age'], [-1, 9.99, 59.99, 100], labels=False
        ),
    }
)

# CV accuracy of CategoricalNB
acc_nb = cross_val_score(CategoricalNB(), X_cat_train, y_train, cv=cv).mean()
print('CategoricalNB: CV accuracy =', round(acc_nb, 3))

**Solution:**

a) `CategoricalNB` gives P(survived | woman, 1st class) ≈ **0.887** – exactly the hand calculation of Exercise 1b.

b) With sex, class and age group, Naive Bayes reaches a CV accuracy of about **0.79** – a little lower than the scaled
k-NN and the RBF SVM (≈ 0.82–0.83), although it uses only three coarse categorical features and is extremely simple and
fast. The independence assumption is violated (e.g. sex and class interact), which costs some accuracy.

## Exercise 6: Reflection

Compare k-NN, SVM and Naive Bayes: for which of the three methods is

a) feature scaling necessary,

b) the training fast but the prediction slow,

c) a very large number of features (e.g. thousands of words) unproblematic?

**Solution:**

a) **k-NN and SVM** are based on distances / inner products → scale the features. Naive Bayes estimates one distribution
per feature and class, scaling is not needed.

b) **k-NN** has practically no training ("lazy learner": it only stores the data), but for each prediction the distances
to all training points must be computed.

c) **Naive Bayes** (one parameter per word and class) and linear **SVMs** (widely used for text classification) cope well
with many features. For k-NN, distances in very high-dimensional spaces become less informative ("curse of
dimensionality").

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')